## Example 1: FastAPI CLI Core Components and Module Discovery

In [ ]:
# Example 1: FastAPI CLI Core Components and Module Discovery
print("fastapi_cli - Core Components and Module Discovery")

# Import and explore fastapi_cli components
import fastapi_cli
from fastapi_cli import cli, discover, exceptions
from fastapi_cli.cli import app as typer_app
from pathlib import Path
import contextlib, tempfile

print(f"FastAPI CLI version: {fastapi_cli.__version__}")
print(f"Module location: {fastapi_cli.__file__}")

# Test module discovery functionality
from fastapi_cli.discover import get_import_data, get_default_path, get_app_name, get_module_data_from_path

# Create a simple FastAPI app in memory for testing
test_app_code = '''
from fastapi import FastAPI
app = FastAPI(title="Test API", version="1.0.0")

@app.get("/")
def read_root():
    return {"message": "Hello World"}
'''

# Write test app to file
test_app_path = Path(tempfile.mkdtemp()) / "test_app.py"
with open(test_app_path, "w") as f:
    f.write(test_app_code)

# Test import data discovery with correct API
try:
    import_data = get_import_data(path=test_app_path, app_name="app")
    print(f"\n📁 Import Data Discovery:")
    print(f"  Import string: {import_data.import_string}")
    print(f"  App name: {import_data.app_name}")
    print(f"  Module import: {import_data.module_data.module_import_str}")
except Exception as e:
    print(f"  Error discovering import data: {e}")
    raise

# Test app name discovery with existing app
try:
    module_data = get_module_data_from_path(test_app_path)
    app_name = get_app_name(mod_data=module_data, app_name="app")  # Use existing app name
    print(f"\n🔤 App Name Discovery: {app_name}")
    assert app_name == "app", app_name
    
    # Show default path discovery: get_default_path() looks in the CURRENT directory for main.py, app.py,
    # api.py (or the same under app/), so run it from a project directory that has a main.py
    with tempfile.TemporaryDirectory() as project_dir:
        (Path(project_dir) / "main.py").write_text(test_app_code)
        with contextlib.chdir(project_dir):
            default_path = get_default_path()
    print(f"📂 Default Path: {default_path}")
    assert default_path == Path("main.py"), default_path
except Exception as e:
    print(f"  Error with app name discovery: {e}")
    raise

print("Example 1 completed")

## Example 2: CLI Commands and Uvicorn Integration

In [ ]:
# Example 2: CLI Commands and Uvicorn Integration
print("fastapi_cli - CLI Commands and Uvicorn Integration")

import uvicorn
from fastapi_cli.cli import run, dev, get_uvicorn_log_config
from fastapi_cli.discover import get_import_data
from pathlib import Path
import tempfile

# Create a sample FastAPI application for testing
sample_app = '''
from fastapi import FastAPI

app = FastAPI(title="Sample API", version="1.0.0")

@app.get("/")
def root():
    return {"message": "Hello from FastAPI CLI!"}

@app.get("/health")
def health():
    return {"status": "healthy"}
'''

# Write sample app to a file in a fresh temporary directory
app_file = Path(tempfile.mkdtemp()) / "sample_app.py"
with open(app_file, "w") as f:
    f.write(sample_app)

print(f"Created sample app: {app_file}")

# Test import data extraction with correct API
try:
    import_data = get_import_data(path=app_file, app_name="app")
    print(f"\n📦 Import Data:")
    print(f"  Import string: {import_data.import_string}")
    print(f"  App name: {import_data.app_name}")
    print(f"  Module import: {import_data.module_data.module_import_str}")
except Exception as e:
    print(f"  Import error: {e}")
    raise

# Test uvicorn log configuration
log_config = get_uvicorn_log_config()
print(f"\n📊 Uvicorn Log Config:")
print(f"  Version: {log_config.get('version', 'N/A')}")
print(f"  Formatters: {list(log_config.get('formatters', {}).keys())}")

# Show CLI functions available
print(f"\n🔧 CLI Functions:")
print(f"  run function: {run}")
print(f"  dev function: {dev}")

print("Example 2 completed")

## Example 3: Logging Configuration and Rich Output

In [ ]:
# Example 3: Logging Configuration and Rich Output
print("fastapi_cli - Logging Configuration and Rich Output")

from fastapi_cli.cli import get_uvicorn_log_config, setup_logging, get_rich_toolkit
from fastapi_cli.cli import logger  # Logger is in cli module, not logging module
import logging

# Test uvicorn logging configuration
log_config = get_uvicorn_log_config()
print(f"📊 Uvicorn Log Configuration:")
print(f"  Version: {log_config['version']}")
print(f"  Disable existing loggers: {log_config['disable_existing_loggers']}")

# Show formatters
formatters = log_config.get('formatters', {})
print(f"\n🎨 Available Formatters:")
for name, config in formatters.items():
    format_str = config.get('format', 'No format specified')
    print(f"  {name}: {format_str[:50]}...")

# Show handlers
handlers = log_config.get('handlers', {})
print(f"\n📝 Available Handlers:")
for name, config in handlers.items():
    handler_class = config.get('class', 'Unknown')
    level = config.get('level', 'INFO')
    print(f"  {name}: class={handler_class}, level={level}")

# Test logging setup
print(f"\n🔧 Setting up logging...")
try:
    setup_logging("info")
    print(f"  ✅ Logging setup completed for 'info' level")
except Exception as e:
    print(f"  ❌ Error setting up logging: {e}")
    raise

# Test FastAPI CLI logger
print(f"\n📋 FastAPI CLI Logger:")
print(f"  Logger name: {logger.name}")
print(f"  Logger level: {logger.level}")
print(f"  Logger handlers: {len(logger.handlers)}")

# Test rich toolkit integration
print(f"\n🌈 Rich Toolkit Integration:")
try:
    rich_toolkit = get_rich_toolkit()
    print(f"  Rich toolkit available: {rich_toolkit is not None}")
    if rich_toolkit:
        print(f"  Console available: {hasattr(rich_toolkit, 'console')}")
except Exception as e:
    print(f"  Error accessing rich toolkit: {e}")
    raise

# Test logging from fastapi_cli.logging module
from fastapi_cli.logging import setup_logging as logging_setup, RichHandler
print(f"\n🎯 FastAPI CLI Logging Module:")
print(f"  Setup logging function: {logging_setup}")
print(f"  RichHandler: {RichHandler}")

print("Example 3 completed")

## Example 4: Working with FastAPI Applications and CLI Functions

In [ ]:
# Example 4: Working with FastAPI Applications and CLI Functions
print("fastapi_cli - Working with FastAPI Applications and CLI Functions")

from fastapi import FastAPI
from fastapi_cli.discover import get_import_data, get_app_name, get_module_data_from_path
from pathlib import Path
import tempfile
import os
import sys

# Create a more complex FastAPI application with multiple apps
complex_app_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(
    title="Complex API",
    description="A more complex FastAPI application",
    version="2.1.0"
)

# Additional app for testing app name discovery
api_v2 = FastAPI(title="API v2", version="2.0.0")

class Item(BaseModel):
    name: str
    price: float
    is_active: bool = True

@app.get("/")
def read_root():
    return {"message": "Complex API is running!"}

@app.get("/items/{item_id}")
def read_item(item_id: int):
    return {"item_id": item_id, "name": "Sample Item"}

@api_v2.get("/")
def read_v2_root():
    return {"message": "API v2 is running!"}
'''

# Write complex app to temporary file
with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
    f.write(complex_app_code)
    complex_app_file = Path(f.name)

print(f"Created complex app: {complex_app_file}")

# Test app name discovery with available apps
app_names = ["app", "api_v2"]  # Use actual app names that exist in the module
print(f"\n🔤 App Name Discovery:")
try:
    module_data = get_module_data_from_path(complex_app_file)
    # get_app_name imports the module, so its folder must be on sys.path (get_import_data adds it itself)
    sys.path.insert(0, str(module_data.extra_sys_path))
    for name in app_names:
        discovered_name = get_app_name(mod_data=module_data, app_name=name)
        print(f"  {name} → {discovered_name}")
except Exception as e:
    print(f"  Error discovering app names: {e}")
    raise

# Test import data extraction with correct API
print(f"\n📦 Import Data Extraction:")
try:
    import_data = get_import_data(path=complex_app_file, app_name="app")
    print(f"  Import string: {import_data.import_string}")
    print(f"  App name: {import_data.app_name}")
    print(f"  Module import: {import_data.module_data.module_import_str}")
    
    # Test with second app
    import_data_v2 = get_import_data(path=complex_app_file, app_name="api_v2")
    print(f"  API v2 Import string: {import_data_v2.import_string}")
except Exception as e:
    print(f"  Error: {e}")
    raise

# Clean up
os.unlink(complex_app_file)
print("Example 4 completed")

## Example 5: Exception Handling and CLI Utilities

In [ ]:
# Example 5: Exception Handling and CLI Utilities
print("fastapi_cli - Exception Handling and CLI Utilities")

from fastapi_cli.exceptions import FastAPICLIException
from rich.tree import Tree  # Tree is from rich, not fastapi_cli
from fastapi_cli.cli import typer, print
import typer as typer_module

# Test FastAPI CLI exception handling
print(f"🚨 Exception Handling:")
print(f"  FastAPICLIException class: {FastAPICLIException}")
print(f"  Exception base: {FastAPICLIException.__bases__}")

# Create and demonstrate exception
try:
    raise FastAPICLIException("Test error message")
except FastAPICLIException as e:
    print(f"  ✅ Caught FastAPICLIException: {e}")

# Test CLI utilities - Tree (from rich)
print(f"\n🌳 CLI Tree Utility:")
print(f"  Tree class: {Tree}")

# Create a tree structure for display
tree = Tree("FastAPI Project")
app_branch = tree.add("app/")
app_branch.add("main.py")
app_branch.add("models.py")

tests_branch = tree.add("tests/")
tests_branch.add("test_main.py")

tree.add("requirements.txt")
tree.add("README.md")

print(f"  Tree structure created with {len(list(tree.children))} top-level items")

# Test typer integration
print(f"\n⚡ Typer Integration:")
print(f"  Typer module: {typer}")
print(f"  Print function: {print}")

# Test version and callback functions
print(f"\n📋 CLI Functions:")
from fastapi_cli.cli import version_callback, callback
print(f"  Version callback: {version_callback}")
print(f"  CLI callback: {callback}")

# Test rich toolkit from fastapi_cli
from fastapi_cli.utils.cli import get_rich_toolkit
try:
    rich_toolkit = get_rich_toolkit()
    print(f"\n🎨 Rich Toolkit:")
    print(f"  Rich toolkit available: {rich_toolkit is not None}")
    if rich_toolkit:
        print(f"  Theme: {rich_toolkit.theme}")
except Exception as e:
    print(f"  Rich toolkit error: {e}")
    raise

print("Example 5 completed")